In [ ]:
from __future__ import annotations

import pathlib
from collections import defaultdict

import numpy as np
import pandas as pd
from kebab.utils.dataset.wikidata.wikidata_utils import ResolvedWikidataEntity

In [ ]:
original_dataset_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Fragments Resolved"
    / "rebel_entity_fragments.jsonl"
)

clustering_dataset_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Clustering Dataset Base"
    / "rebel_clustering_dataset.jsonl"
)

dataset_path = clustering_dataset_path

Load the fragments

In [ ]:
fragments = []

# we will not be using fragments with no names
fragments_with_no_names = 0

# load the data
with open(dataset_path, encoding="utf-8") as f:
    for line in f:
        fragment = ResolvedWikidataEntity.from_json(line.strip())

        if not fragment.names:
            fragments_with_no_names += 1
            del fragment
            continue

        # reduce memory footprint
        if "doc_id" in fragment.metadata:
            del fragment.metadata["doc_id"]

        if "source_text_hash" in fragment.metadata:
            del fragment.metadata["source_text_hash"]

        if "fragment_id" in fragment.metadata:
            del fragment.metadata["fragment_id"]

        fragment.evidence_map = None
        fragment.source_ids = None

        fragments.append(fragment)

print(f"Loaded {len(fragments):,d} fragments, ignored {fragments_with_no_names:,d} fragments with no names")

Example fragment

In [ ]:
fragments[0]

In [ ]:
# compute counts of property occurrence and entity types of the fragments
property_counts = defaultdict(int)
type_counts = defaultdict(int)

for fragment in fragments:
    for prop_name, prop_value in fragment.properties.items():
        if prop_value:
            property_counts[prop_name] += 1

    for ent_type in fragment.wikidata_type:
        type_counts[ent_type] += 1

Top properties by occurrence in the fragments
---

In [ ]:
df = (
    pd.DataFrame(property_counts.items(), columns=["property", "count"])
    .sort_values(by="count", ascending=False)
    .reset_index(drop=True)
)
df.to_csv("property_occurrence.csv", index=False)
df[:20]

Top entity types of the fragments
---

In [ ]:
df = (
    pd.DataFrame(type_counts.items(), columns=["type", "count"])
    .sort_values(by="count", ascending=False)
    .reset_index(drop=True)
)
df.to_csv("type_occurrence.csv", index=False)
df[:20]

Properties overlap
---

In [ ]:
# for each property how often that two distinct fragments have (1) a value for this property, and (2) the same value this property
property_value_counts = defaultdict(lambda: defaultdict(int))
entity_value_counts = defaultdict(int)

for entity in fragments:
    for prop_name, values in entity.properties.items():
        entity_value_counts[prop_name] += 1
        for value in values:
            property_value_counts[prop_name][value] += 1

entity_count = len(fragments)

rows = []
for prop_name, value_counts in property_value_counts.items():
    arr = np.array(list(value_counts.values()))
    ent_val_count = entity_value_counts[prop_name]
    ent_probs = arr / entity_count
    cond_ent_probs = arr / ent_val_count
    prob = (ent_probs**2).sum()
    cond_prob = (cond_ent_probs**2).sum()
    ent_fraction = ent_val_count / entity_count
    rows.append((prop_name, len(value_counts), prob, cond_prob, ent_fraction))

overlap_df = pd.DataFrame(
    rows, columns=["property", "distinct_value_count", "overlap_prob", "cond_overlap_prob", "entities_fraction"]
)
overlap_df = overlap_df.sort_values("overlap_prob", ascending=False)
overlap_df.head(100)